# 07 · Two-stage stacked ensemble (X4), final model and submission

Stage 1 is a set of diverse base models trained on the train part: word TF-IDF (B0), per-language (X1), per-domain (X2g), the label cascade (X5), character n-grams (X6), the sentence model (X7), the encoder (E1, with its window statistics), the token-level encoder (E2, with its per-text token summaries) and the LM-feature classifier (X3). Stage 2 is a gradient-boosting stacker per language whose inputs are their validation/test outputs, the given `domain` column (present in test) and text statistics. Raw LM features are switched off (`lm_features: false`); X3's scores carry the same signal. The stage-1 domain classifier is not used: X2 vs X2g in `06` shows that predicting the domain only matches the given column.

The stacker only sees validation rows: it is scored with out-of-fold predictions from a 5-fold `StratifiedGroupKFold` over the validation split (groups from `02_split`), so it is never evaluated on rows it was fitted on; for the test set it is refit on all validation rows. Members missing on disk are skipped. X4b adds per-language class offsets to the stacker's log-probabilities, tuned for macro-F1; the offsets for each validation fold come from the other folds.

The final model is the one with the highest validation macro-F1 in the experiment log.

In [ ]:
import json
import warnings

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.exceptions import ConvergenceWarning

from hw3.config import ROOT, art, load_config

warnings.filterwarnings("ignore", category=ConvergenceWarning)
pd.set_option("display.width", 200, "display.max_columns", 40)
FIG = ROOT / "images"

from hw3 import ensemble
from hw3.runs import compare, error_overlap, experiment_log, report
from hw3.submission import save_submission, select_final

In [ ]:
Xv, oof = ensemble.run_ensemble(load_config("ensemble"))
meta = json.loads(art("runs", "X4.json").read_text())
meta["members"], Xv.shape, json.loads(art("runs", "X4b.json").read_text())["offsets"]

In [ ]:
rep = report("X4")
rep["figure"].savefig(FIG / "confusion_X4.png", dpi=150, bbox_inches="tight")
rep["per_class"]

In [ ]:
compare(["B0", "X1", "X2g", "X5", "X6", "X7", "E1", "E2", "X3", "X4", "X4b"])

Error overlap of the members on the validation split: an ensemble gains where one member is wrong and another is right (`only_one_wrong`), not where both are wrong.

In [ ]:
error_overlap(meta["members"])

## Final model selection

In [ ]:
log = experiment_log()
best = select_final(log)
log

In [ ]:
best

## Submission with automated checks

The checks run on the frame before writing and again on the file read back from disk; any failure raises and nothing is uploaded.

In [ ]:
save_submission(best.id)

Upload manually after the checks pass:

```bash
kaggle competitions submit -c ai-generated-text-detection-hw3-ucu-ai-2026 -f submission.csv -m "<model id>"
```